# 문제 1 — 벡터 연산 모듈 (내적·외적·정사영·rank)

로봇의 좌표 변환은 결국 **벡터 연산**의 조합입니다. 이 노트북에서는
내적·사이각·정규화·정사영·반대칭행렬(외적)·평면 법선·rank 를
`np.linalg` 없이 직접 구현하고, 각각을 검증합니다.

완성한 함수는 `src/vectors.py` 에 채워 넣어 문제 2 이후에서 재사용합니다.

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 1-1 | 내적과 사이각을 구하고 **손계산 값과 일치**하는지 검증 | `dot`, `norm`, `angle_between` |
| 1-2 | 정규화 함수를 만들고 **영벡터를 넣으면 어떻게 되는지 직접 실행해 기록**한 뒤 처리 방식을 정해 구현 | `normalize` |
| 1-3 | 정사영을 구현하고 ① 남는 성분이 수직인지 ② 두 성분의 합이 원래 벡터인지 검증 | `project`, `reject` |
| 1-4 | 외적을 **반대칭행렬 곱**으로 구현하고 `np.cross` 와 비교, 반대칭성 검증 | `skew`, `cross` |
| 1-5 | 세 점이 만드는 평면의 **단위 법선** | `plane_normal` |
| 1-6 | (1,0,1), (0,1,1), (1,1,2) 의 rank 를 구하고 **왜 3 이 아닌지** 설명, 행렬식과 일관성 확인 | `row_echelon`, `rank`, `det` |

> **규약**
> - 난수는 `np.random.default_rng(42)` 로 고정합니다.
> - 수치 비교는 부동소수점 오차를 고려해 `np.allclose` / `np.isclose` 로 합니다.
> - `np.linalg` 는 **검산용으로만** 쓰고, 쓸 때마다 주석으로 검산임을 밝힙니다.
> - 각 문항은 **(1) 설명 마크다운 → (2) 코드 → (3) 검증** 순서를 지킵니다.

In [15]:
import sys
import os

from pathlib import Path

import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(),"..")))
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.vectors import (angle_between, cross, det, dot, norm, normalize,
                         plane_normal, project, rank, reject, row_echelon, skew)

rng = np.random.default_rng(42)          # 시드 고정
np.set_printoptions(precision=6, suppress=True)


def check(label, condition):
    """검증 셀에서 쓰는 통과/실패 출력 헬퍼. (그대로 쓰면 됩니다)"""
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


print("NumPy", np.__version__)

NumPy 2.2.6


## 1-1. 내적과 사이각

내적의 정의는 두 가지이며 서로 같습니다.

$$\mathbf{a}\cdot\mathbf{b}=\sum_i a_i b_i = |\mathbf{a}||\mathbf{b}|\cos\theta$$

두 번째 식을 $\theta$ 에 대해 풀면 사이각이 나옵니다.

검증하기 쉽도록 **손으로 계산되는 값**을 고릅니다.
$\mathbf{a}=(3,4,0)$, $\mathbf{b}=(4,3,0)$ 이면 $|\mathbf{a}|=|\mathbf{b}|=5$ 이므로
내적과 $\cos\theta$, 사이각을 종이에서 먼저 구한 뒤 코드 결과와 비교하세요.

**할 일** — `src/vectors.py` 의 `dot`, `norm`, `angle_between` 을 구현하고
아래 셀에서 손계산 값과 나란히 출력합니다.

In [16]:
a = np.array([3.0, 4.0, 0.0])
b = np.array([4.0, 3.0, 0.0])

# TODO: dot / norm / angle_between 을 호출해 아래 값을 구하고 출력하세요.
d          = dot(a,b)
theta_deg  = angle_between(a,b)
# TODO: 손으로 계산한 값(hand_dot, hand_cos, hand_deg)도 함께 출력해 비교하세요.

hand_dot = 3*4+4*3+0*0
hand_cos = 24/(5*5)
hand_deg = 16.2602

In [17]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 내적이 손계산 값과 일치하는가
#   - 사이각이 손계산 값과 일치하는가
#   - np.dot 검산 결과와 일치하는가            (# 검산용 이라고 주석을 남길 것)
#   - 수직인 두 벡터의 사이각이 90도인가
#   - 같은 벡터끼리의 사이각이 0도인가
check("내적이 손계산 값과 일치하는가", np.isclose(d, hand_dot))
check("사이각이 손계산 값과 일치하는가", np.isclose(theta_deg, hand_deg))
check("np.dot 검산 결과와 일치하는가", np.isclose(d, np.dot(a, b))) # [검산] 내장함수 비교명시
check("수직인 두 벡터의 사이각이 90도인가", np.isclose(90, angle_between(np.array([0,1]),np.array([1,0]))))
check("같은 벡터끼리의 사이각이 0도인가", np.isclose(0, angle_between(a,a)))

[PASS] 내적이 손계산 값과 일치하는가
[PASS] 사이각이 손계산 값과 일치하는가
[PASS] np.dot 검산 결과와 일치하는가
[PASS] 수직인 두 벡터의 사이각이 90도인가
[PASS] 같은 벡터끼리의 사이각이 0도인가


True

## 1-2. 정규화와 영벡터 — 무슨 일이 일어나는가

정규화는 $\hat{\mathbf{v}} = \mathbf{v}/|\mathbf{v}|$ 입니다.

**할 일**

1. 먼저 **아무 보호 장치 없이** 영벡터를 길이로 나눠 보고, 실제로 무엇이 출력되는지
   (경고 메시지 포함) 그대로 기록하세요.
   경고를 죽이고 관찰하려면 `with np.errstate(invalid="ignore", divide="ignore"):` 를 쓰면 됩니다.
2. 그 결과가 **왜 위험한지** 생각해 보세요. 이후 연산에 어떻게 전파되는지,
   `assert` 나 `==` 비교로 잡히는지 직접 확인해 보면 답이 보입니다.
3. 어떻게 처리할지 **직접 정하고**(예: 예외를 던진다 / 영벡터를 그대로 돌려준다 /
   특정 축을 돌려준다 …) `src/vectors.py` 의 `normalize` 에 구현하세요.
4. 아래 마크다운에 **선택한 방식과 근거**를 적으세요. 검증 셀도 그 방식에 맞춰 작성합니다.

### 선택한 처리 방식과 근거

- 관찰한 결과: `nan`
- 선택한 처리: `영벡터를 그대로 돌려준다`
- 근거: `nan==nan도 False처리되어 예외처리해야하는데 0으로 나눈 결과가 0인 것 자체는 오류지만 다른 예외처리보다 코드도 간단하고 자연스러운 오류처리가 가능함`

In [18]:
zero = np.array([0.0, 0.0, 0.0])

# TODO: (1) 보호 없이 나눴을 때 무슨 값이 나오는지 관찰해 출력하세요.
norm_zero = normalize(zero)
with np.errstate(invalid="ignore", divide="ignore"):
    normalized_zero = zero / norm_zero
print(normalized_zero)
# TODO: (2) 그 값이 이후 비교/전파에서 어떻게 동작하는지 확인해 출력하세요.
assert np.all(np.isnan(normalized_zero))
print(normalized_zero==np.nan)
# TODO: (3) 구현한 normalize 로 정상 벡터와 영벡터를 각각 처리해 출력하세요.
print(normalize(a))
print(np.isnan(norm_zero))

[nan nan nan]
[False False False]
[0.6 0.8 0. ]
[False False False]


In [19]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 보호 없이 나눴을 때 관찰한 현상이 실제로 재현되는가
check("보호 없이 나눴을 때 nan 발생 여부", np.all(np.isnan(normalized_zero)))
#   - 정규화된 벡터의 길이가 1 인가
a_hat = normalize(a)
check("정규화된 벡터의 길이가 1인지 확인", np.isclose(np.linalg.norm(a_hat), 1.0))
#   - 정규화가 방향을 바꾸지 않는가 (원본과의 사이각이 0)
cos_theta = np.dot(a, a_hat) / (np.linalg.norm(a) * np.linalg.norm(a_hat))
check("정규화 후 방향 유지 여부", np.isclose(cos_theta, 1.0))
#   - 영벡터 입력에서 내가 정한 처리 방식대로 동작하는가
check("영벡터 입력 시 영벡터 반환 여부", np.allclose(norm_zero, np.array([0.0, 0.0, 0.0])))
#   - 무작위 벡터도 정규화 후 길이가 1 인가
rand_v = rng.uniform(-10.0, 10.0, size=3)
if np.linalg.norm(rand_v) == 0: rand_v = np.array([1.0, 1.0, 1.0]) # 예외 방지
rand_v_hat = normalize(rand_v)
check("무작위 벡터 정규화 후 길이 1 확인", np.isclose(np.linalg.norm(rand_v_hat), 1.0))

[PASS] 보호 없이 나눴을 때 nan 발생 여부
[PASS] 정규화된 벡터의 길이가 1인지 확인
[PASS] 정규화 후 방향 유지 여부
[PASS] 영벡터 입력 시 영벡터 반환 여부
[PASS] 무작위 벡터 정규화 후 길이 1 확인


True

## 1-3. 정사영 — 수직성과 합 복원

$\mathbf{a}$ 를 $\mathbf{b}$ 방향으로 정사영한 성분은

$$\mathrm{proj}_{\mathbf{b}}(\mathbf{a})=\frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{b}\cdot\mathbf{b}}\mathbf{b}$$

이고, 남는 성분(reject)은 $\mathbf{a}-\mathrm{proj}_{\mathbf{b}}(\mathbf{a})$ 입니다.
분모가 $|\mathbf{b}|^2$ 이므로 $\mathbf{b}$ 를 미리 정규화할 필요가 없습니다.

**검증해야 할 두 가지**

1. **수직성**: (남는 성분) $\cdot$ $\mathbf{b} = 0$
2. **합 복원**: $\mathrm{proj} + \mathrm{rej} = \mathbf{a}$

In [20]:
a = np.array([2.0, 3.0, 4.0])
b = np.array([1.0, 0.0, 1.0])

# TODO: project / reject 를 호출하고, 계수 (a·b)/(b·b) 와 함께 결과를 출력하세요.
proj_v = project(a,b)
rej_v = reject(a,b)
coefficient = dot(a,b)/dot(b,b)
print(coefficient, proj_v, rej_v)

3.0 [3. 0. 3.] [-1.  3.  1.]


In [21]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - ① 남는 성분이 b 와 수직인가 (rej·b = 0)
vertical = np.isclose(dot(rej_v,b), 0.0)
check("남는 성분이 b 와 수직인가 (rej·b = 0)", vertical)
#   - ② proj + rej = a 인가
reconstruction = np.all(np.isclose(proj_v+rej_v, a))
check("proj + rej = a 인가", reconstruction)
#   - proj 가 b 와 평행한가 (외적이 0)
check("proj 가 b 와 평행한가 (외적이 0)", np.all(np.isclose(cross(proj_v,b), 0.0)))
#   - 피타고라스: |a|^2 = |proj|^2 + |rej|^2
check("피타고라스: |a|^2 = |proj|^2 + |rej|^2", np.isclose(norm(a)**2, norm(proj_v)**2+norm(rej_v)**2))
#   - 무작위 100 쌍에서도 ①②가 모두 성립하는가  (rng 로 생성)
all_passed = True

for i in range(100):
    rand_a = rng.uniform(-10.0, 10.0, size=3)
    rand_b = rng.uniform(-10.0, 10.0, size=3)

    if np.linalg.norm(rand_b) == 0:
        rand_b = np.array([1.0, 1.0, 1.0])

    r_proj = project(rand_a, rand_b)
    r_rej = reject(rand_a,rand_b)

    cond1=np.isclose(dot(r_rej, rand_b), 0.0)
    cond2=np.allclose(r_proj+r_rej, rand_a)

    if not cond1 or not cond2:
        all_passed=False
        break

check("무작위 100 쌍에서도 ①②가 모두 성립하는가  (rng 로 생성)", all_passed)

[PASS] 남는 성분이 b 와 수직인가 (rej·b = 0)
[PASS] proj + rej = a 인가
[PASS] proj 가 b 와 평행한가 (외적이 0)
[PASS] 피타고라스: |a|^2 = |proj|^2 + |rej|^2
[PASS] 무작위 100 쌍에서도 ①②가 모두 성립하는가  (rng 로 생성)


True

## 1-4. 외적을 반대칭행렬 곱으로 — `skew(a)`

외적은 행렬 곱으로 쓸 수 있습니다.

$$\mathbf{a}\times\mathbf{b} = [\mathbf{a}]_\times \mathbf{b},\qquad
[\mathbf{a}]_\times=\begin{bmatrix}0&-a_3&a_2\a_3&0&-a_1\-a_2&a_1&0\end{bmatrix}$$

이 형태가 중요한 이유는 **로드리게스 공식(문제 2)과 각속도 → 회전 미분**이
전부 $[\boldsymbol{\omega}]_\times$ 로 표현되기 때문입니다.
반대칭(skew-symmetric)이란 $M^{\mathsf{T}} = -M$ 을 뜻합니다.
여기서 따라오는 성질이 하나 더 있는데, 직접 출력해서 확인해 보세요.

In [22]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# TODO: skew(a) 를 출력하고, skew(a) @ b 와 np.cross(a, b)(# 검산용)를 비교하세요.
s = skew(a)
skew_cross = s @ b
assert np.allclose(skew_cross, np.cross(a,b))
# TODO: skew(a).T 와 -skew(a) 를 나란히 출력해 반대칭성을 눈으로 확인하세요.
print("skew(a).T 와 -skew(a) 를 나란히 출력", s.T, -s)
assert np.allclose(s.T, -s)

skew(a).T 와 -skew(a) 를 나란히 출력 [[ 0.  3. -2.]
 [-3.  0.  1.]
 [ 2. -1.  0.]] [[-0.  3. -2.]
 [-3. -0.  1.]
 [ 2. -1. -0.]]


In [23]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - skew(a) @ b == np.cross(a, b)
check("skew(a) @ b == np.cross(a, b)",np.allclose(skew_cross, np.cross(a,b)))
#   - skew(a) 가 반대칭인가 (S.T == -S)
check("skew(a) 가 반대칭인가 (S.T == -S)",np.allclose(s.T, -s))
#   - 반대칭에서 따라오는 대각성분 성질
check("반대칭에서 따라오는 대각성분 성질", np.allclose(np.diag(s), 0))
#   - skew(a) @ a == 0 (자기 자신과의 외적)
check("skew(a) @ a == 0 (자기 자신과의 외적)", np.allclose(s @ a, 0))
#   - 반교환성: a x b == -(b x a)
check("반교환성: a x b == -(b x a)", np.allclose(cross(a,b), -cross(b,a)))
#   - 무작위 200 쌍에서 skew 곱 == np.cross
for i in range(200):
    rand_a = rng.uniform(-10.0, 10.0, size=3)
    rand_b = rng.uniform(-10.0, 10.0, size=3)

    rand_c = skew(rand_a) @ rand_b

    cond=np.allclose(rand_c, np.cross(rand_a,rand_b))

    if not cond:
        all_passed=False
        break

check("무작위 200 쌍에서 skew 곱 == np.cross", all_passed)

[PASS] skew(a) @ b == np.cross(a, b)
[PASS] skew(a) 가 반대칭인가 (S.T == -S)
[PASS] 반대칭에서 따라오는 대각성분 성질
[PASS] skew(a) @ a == 0 (자기 자신과의 외적)
[PASS] 반교환성: a x b == -(b x a)
[PASS] 무작위 200 쌍에서 skew 곱 == np.cross


True

## 1-5. 세 점이 만드는 평면의 단위 법선

세 점 $P_1,P_2,P_3$ 이 주어지면 두 모서리 벡터
$\mathbf{u}=P_2-P_1$, $\mathbf{v}=P_3-P_1$ 의 외적이 평면에 수직입니다.
이를 정규화하면 단위 법선입니다.

세 점이 **일직선**이면 어떻게 될지 먼저 생각해 보고, 그 경우를 어떻게 처리할지 정하세요.

In [24]:
P1 = np.array([0.0, 0.0, 0.0])
P2 = np.array([1.0, 0.0, 0.0])
P3 = np.array([0.0, 1.0, 0.0])

# TODO: xy 평면 위 세 점의 단위 법선을 구해 출력하고, 기대값과 비교하세요.
v21 = P2 - P1
v31 = P3 - P1

n_xy=cross(v21,v31)
n_xy_norm = normalize(n_xy)

print(n_xy_norm)

# TODO: 기울어진 평면(예: (1,0,0), (0,1,0), (0,0,1))에서도 구해 보세요.
X = np.array([1.0,0.0,0.0])
Y = np.array([0.0,1.0,0.0])
Z = np.array([0.0,0.0,1.0])

v_yx = Y - X
v_zx = Z - X

n_tilt = cross(v_yx, v_zx)
n_tilt_norm = normalize(n_tilt)
print(n_tilt_norm)
# TODO: 일직선인 세 점을 넣으면 어떻게 되는지 확인해 출력하세요.
L1=np.array([0.0,0.0,0.0])
L2=np.array([1.0,0.0,0.0])
L3=np.array([2.0,0.0,0.0])

v_L2 = L2-L1
v_L3 = L3-L1

L_cross=cross(v_L2,v_L3)
L_cross_norm = normalize(L_cross)

print(L_cross_norm)

[0. 0. 1.]
[0.57735 0.57735 0.57735]
[0. 0. 0.]


In [25]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 법선의 길이가 1 인가
check("법선의 길이가 1 인가", np.isclose(norm(n_xy_norm), 1))
#   - xy 평면의 법선이 z축과 일치하는가
check("xy 평면의 법선이 z축과 일치하는가", np.allclose(n_xy_norm,np.array([0.0,0.0,1.0])))
#   - 법선이 두 모서리 벡터 모두와 수직인가
cond_v21=np.isclose(dot(n_xy_norm, v21), 0.0)
cond_v31=np.isclose(dot(n_xy_norm, v31), 0.0)
check("법선이 두 모서리 벡터 모두와 수직인가", cond_v21 and cond_v31)
#   - 기울어진 평면의 법선이 기대값과 일치하는가
expected_tilt = normalize(np.array([1.0, 1.0, 1.0]))
check("기울어진 평면의 법선이 기대값과 일치하는가", np.allclose(expected_tilt,n_tilt_norm))
#   - 일직선 입력에서 내가 정한 처리 방식대로 동작하는가
check("일직선 입력에서 내가 정한 처리 방식대로 동작하는가", np.allclose(L_cross_norm, np.array([0.0, 0.0, 0.0])))

[PASS] 법선의 길이가 1 인가
[PASS] xy 평면의 법선이 z축과 일치하는가
[PASS] 법선이 두 모서리 벡터 모두와 수직인가
[PASS] 기울어진 평면의 법선이 기대값과 일치하는가
[PASS] 일직선 입력에서 내가 정한 처리 방식대로 동작하는가


True

## 1-6. rank 와 행렬식 — 왜 3 이 아닌가

세 벡터 $(1,0,1)$, $(0,1,1)$, $(1,1,2)$ 를 행으로 쌓은 행렬의 rank 를 구합니다.
rank 는 **선형독립인 행(또는 열)의 개수**이고, 행 사다리꼴로 만들었을 때
살아남는 피벗의 개수와 같습니다.

코드를 돌리기 **전에** 세 벡터를 눈으로 보고 서로 어떤 관계인지 찾아보세요.
그 관계가 곧 "왜 3 이 아닌가" 의 답입니다.
정사각 행렬에서 rank 와 행렬식은 서로 일관돼야 한다는 점도 확인합니다.

**할 일** — `row_echelon`, `rank`, `det` 를 구현하고 아래를 채우세요.

### rank 가 3 이 아닌 이유

- 발견한 선형종속 관계: `(1, 0, 1) + (0, 1, 1) = (1, 1, 2)`
- 세 벡터가 span 하는 공간: `세 벡터 중 독립적인 벡터가 2개뿐이기 때문에 2차원 평면만을 형성`
- rank 와 행렬식이 일관되는 이유: `선형종속인 행렬의 행렬식은 항상 0이기 때문`

In [26]:
M = np.array([[1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0],
              [1.0, 1.0, 2.0]])

# TODO: row_echelon 으로 사다리꼴과 피벗 열을 출력하세요.
ref_M = row_echelon(M)
print("사다리꼴")
for row in ref_M:
    print([round(val, 4) for val in row])

print("피벗 열")
pivot_cols = []
for r in range(len(ref_M)):
    for c in range(len(ref_M[0])):
        if abs(ref_M[r][c]) >= 1e-9:
            pivot_cols.append(c)
            break

print(pivot_cols)
# TODO: 직접 구현한 rank / det 를 np.linalg.matrix_rank / np.linalg.det (# 검산용) 와 비교하세요.
my_rank = rank(M)
np_rank = np.linalg.matrix_rank(M)
my_det = det(M.tolist())
np_det = np.linalg.det(M)

assert np.allclose(my_rank,np_rank)
assert np.isclose(my_det, np_det)
# TODO: 세 벡터 사이의 선형종속 관계를 코드로 확인해 출력하세요.
#       (스칼라 삼중곱 dot(v1, cross(v2, v3)) 도 같이 보면 좋습니다)
print("선형종속 관계 확인")

M_v1 = M[0]
M_v2 = M[1]
M_v3 = M[2]
linear_dependency = M_v1 + M_v2 - M_v3
print("v1+v2=v3 : ",np.allclose(linear_dependency, 0.0))

triple_product = dot(M_v1, cross(M_v2, M_v3))
print("삼중곱 : ", triple_product)
print("삼중곱이 0인가 : ", np.isclose(triple_product, 0.0))

사다리꼴
[1.0, 0.0, 1.0]
[0.0, 1.0, 1.0]
[0.0, 0.0, 0.0]
피벗 열
[0, 1]
선형종속 관계 확인
v1+v2=v3 :  True
삼중곱 :  0.0
삼중곱이 0인가 :  True


In [27]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - rank 가 3 이 아닌 값인가
check("rank 가 3 이 아닌 값인가", my_rank != 3)
#   - 직접 구현 rank == np.linalg.matrix_rank
check("직접 구현 rank == np.linalg.matrix_rank", np.allclose(my_rank, np_rank))
#   - 찾아낸 선형종속 관계가 실제로 성립하는가
check("찾아낸 선형종속 관계가 실제로 성립하는가", np.allclose(linear_dependency, 0.0))
#   - 행렬식이 0 인가 / 직접 구현 det == np.linalg.det
check("행렬식이 0 인가 / 직접 구현 det == np.linalg.det", np.isclose(my_det, 0) and np.allclose(my_det,np_det))
#   - rank < 3 과 det == 0 이 서로 일관되는가
for _ in range(100):
    rand_M = rng.uniform(-5.0,5.0,size=(3,3))

    if rng.uniform() > 0.5:
        rand_M[2] = rand_M[0] + rand_M[1]

    r_rank = rank(rand_M)
    r_det = det(rand_M.tolist())

    cond_rank_deficient = (r_rank < 3)
    cond_det_zero = np.isclose(r_det, 0.0)

    if cond_rank_deficient != cond_det_zero:
        all_passed = False
        break

check("rank < 3 과 det == 0 이 서로 일관되는가", all_passed)
#   - 반례: 단위행렬은 rank 3, det 1 인가
I = np.eye(3)
check("반례: 단위행렬은 rank 3, det 1 인가", rank(I) == 3 and np.isclose(det(I.tolist()), 1.0))

[PASS] rank 가 3 이 아닌 값인가
[PASS] 직접 구현 rank == np.linalg.matrix_rank
[PASS] 찾아낸 선형종속 관계가 실제로 성립하는가
[PASS] 행렬식이 0 인가 / 직접 구현 det == np.linalg.det
[PASS] rank < 3 과 det == 0 이 서로 일관되는가
[PASS] 반례: 단위행렬은 rank 3, det 1 인가


True

## 답안 템플릿 정리

지시문의 답안 템플릿에 맞춰 아래 빈칸을 채워 출력하세요.
값은 위에서 계산한 변수를 그대로 넣고, 설명은 직접 문장으로 씁니다.

In [28]:
summary = f"""
1. 내적: {d:.4f} / 사이각: {theta_deg:.2f} 도
   - 손계산과 일치 여부: {np.allclose(d, hand_dot) and np.allclose(theta_deg,hand_deg)}

2. 영벡터 정규화 시 결과: {norm_zero}
   - 선택한 처리: 영벡터 입력시 그대로 영백터를 반환하도록 처리
   - 근거: nan==nan도 False처리되어 예외처리해야하는데 0으로 나눈 결과가 0인 것 자체는 오류지만 다른 예외처리보다 코드도 간단하고 자연스러운 오류처리가 가능함

3. 정사영 검증: 수직성 {vertical} / 합 복원 {reconstruction}

4. skew(a) @ b 와 np.cross(a, b) 일치: {np.allclose(skew_cross, np.cross(a,b))}

5. 세 벡터의 rank: {my_rank}
   - 3 이 아닌 이유: v1 + v2 = v3의 선형종속 관계 때문
   - 행렬식 값: {my_det:.4f}  -> rank 와 일관되는가: {(my_rank < 3) == np.isclose(my_det, 0.0)}
"""
print(summary)


1. 내적: 24.0000 / 사이각: 16.26 도
   - 손계산과 일치 여부: True

2. 영벡터 정규화 시 결과: [0. 0. 0.]
   - 선택한 처리: 영벡터 입력시 그대로 영백터를 반환하도록 처리
   - 근거: nan==nan도 False처리되어 예외처리해야하는데 0으로 나눈 결과가 0인 것 자체는 오류지만 다른 예외처리보다 코드도 간단하고 자연스러운 오류처리가 가능함

3. 정사영 검증: 수직성 True / 합 복원 True

4. skew(a) @ b 와 np.cross(a, b) 일치: True

5. 세 벡터의 rank: 2
   - 3 이 아닌 이유: v1 + v2 = v3의 선형종속 관계 때문
   - 행렬식 값: 0.0000  -> rank 와 일관되는가: True

